In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# hide future warnings for pandas
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

SMALL_FONT_SIZE = 12
MEDIUM_FONT_SIZE = 16
LARGE_FONT_SIZE = 20
XLARGE_FONT_SIZE = 24

plt.rc("font", size=MEDIUM_FONT_SIZE)
plt.rc("axes", titlesize=LARGE_FONT_SIZE)
plt.rc("axes", labelsize=LARGE_FONT_SIZE)
plt.rc("xtick", labelsize=MEDIUM_FONT_SIZE)
plt.rc("ytick", labelsize=MEDIUM_FONT_SIZE)
plt.rc("legend", fontsize=MEDIUM_FONT_SIZE)
plt.rc("figure", titlesize=LARGE_FONT_SIZE)

# Define a function for plotting the main metric for an evaluation task

In [ ]:
def plot_main_metric(models, dataset, evaluation, task, include_spikeins, include_downsampling_schemes,
                     y_lim_low=0.3, suffix=""):
    sns.set_style("ticks")
    
    fig, axes = plt.subplots(1, len(models), figsize=(24, 6), sharey=False)

    for i, model in enumerate(models):
        ax = axes[i]

        print("plotting:")
        print(evaluation, model, dataset)

        if task == "classification":
            metric = "micro_f1"
            y_label = "Micro F1 Score"
        if task == "integration":
            metric = "avg_bio"
            y_label = "Avg. BIO"

        if task == "perturbation":
            metric = "R2"
            y_label = "$R^2$"

        if evaluation == "finetune" and task == "classification":
            metrics_df = pd.read_csv(f"metrics_csvs/{model}_finetune_classification_eval_results.csv", index_col=0)
            logistic_regression_baseline_df = pd.read_csv(f"metrics_csvs/finetune_classification_logistic_regression_variable_genes_baselines.csv", index_col=0)
            logistic_regression_baseline_df = logistic_regression_baseline_df[logistic_regression_baseline_df.dataset == dataset]

            logistic_regression_baseline = logistic_regression_baseline_df["micro_f1"][0]

        if evaluation == "zeroshot" and task == "classification":
            metrics_df = pd.read_csv(f"metrics_csvs/{model}_zeroshot_classification_eval_results.csv", index_col=0)
            hvg_baseline_df = pd.read_csv(f"metrics_csvs/zero_shot_classification_variable_genes_baselines.csv", index_col=0)
            pca_baseline_df = pd.read_csv(f"metrics_csvs/zero_shot_classification_pca_baselines.csv", index_col=0)

            
        if evaluation == "zeroshot" and task == "integration":
            metrics_df = pd.read_csv(f"metrics_csvs/{model}_zeroshot_integration_eval_results.csv", index_col=0)

            hvg_baseline_df = pd.read_csv(f"metrics_csvs/zero_shot_integration_variable_genes_baselines.csv", index_col=0)
            pca_baseline_df = pd.read_csv(f"metrics_csvs/zero_shot_integration_pca_baselines.csv", index_col=0)

        if evaluation == "finetune" and task == "perturbation":
            metrics_df = pd.read_csv(f"metrics_csvs/{model}_finetune_perturbation_eval_results.csv", index_col=0)
            perturbation_baseline_df = pd.read_csv("metrics_csvs/finetune_perturbation_noprediction_baseline.csv", index_col=0)
            perturbation_baseline_df = perturbation_baseline_df[perturbation_baseline_df.dataset == dataset]

            perturbation_baseline = perturbation_baseline_df["R2"][0]
        
        # subset to the dataset of interest
        metrics_df = metrics_df[metrics_df.dataset == dataset]

        # exclude coresets
        metrics_df = metrics_df[metrics_df.downsampling_method != "coresets"]

        # subset to remove downsampling schemes for spikein supplemental figures
        if not include_downsampling_schemes:
            metrics_df = metrics_df[metrics_df.downsampling_method != "geometric_sketching"]
            metrics_df = metrics_df[metrics_df.downsampling_method != "celltype_reweighted"]

        # subset to remove spikeins for main text figures
        if not include_spikeins:
            metrics_df = metrics_df[metrics_df.downsampling_method != "spikein_10"]
            metrics_df = metrics_df[metrics_df.downsampling_method != "spikein_50"]


        if evaluation == "zeroshot":
            hvg_baseline_df = hvg_baseline_df[hvg_baseline_df.dataset == dataset]
            pca_baseline_df = pca_baseline_df[pca_baseline_df.dataset == dataset]
            
            hvg_baseline = hvg_baseline_df[metric][0]
            pca_baseline = pca_baseline_df[metric][0]


        # process 0pcts differently
        # put the untrained results in their own df
        untrained_baseline_df = metrics_df[metrics_df.percentage == 0]
        metrics_df = metrics_df[metrics_df.percentage != 0]


        # make it so that untrained results line will go across whole plot
        untrained_baseline_df_copy = untrained_baseline_df.copy()
        untrained_baseline_df_copy['percentage'] = 100
        untrained_baseline_df = pd.concat([untrained_baseline_df, untrained_baseline_df_copy], axis=0)
        untrained_baseline_df['downsampling_method'] = "Non pre-trained"

        
        # add the untrained results back to the original df
        metrics_df = pd.concat([metrics_df, untrained_baseline_df], axis=0)


        # rename categories for plotting
        metrics_df["downsampling_method"] = metrics_df["downsampling_method"].replace({
            'randomsplits': 'Random', 
            'random': 'Random', 
            'celltype_reweighted': 'Cell Type\nReweighted',
            'geometric_sketch': "Geometric Sketching",
            'geometric_sketching': "Geometric Sketching",
            'spikein_10': "Spike-in (10%)",
            'spikein_50': "Spike-in (50%)",
            })

        LINEWIDTH = 3

        # default matplotlib colors:
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
        
        color_map = {
            'Random': colors[2],
            'Cell Type\nReweighted': colors[0],
            'Geometric Sketching': colors[1],
            'Spike-in (10%)': colors[3],
            'Spike-in (50%)': colors[4],
            'Non pre-trained': colors[5]
        }
        
        # main plot
        sns.lineplot(x='percentage',
                     y=metric,
                     hue='downsampling_method',
                     data=metrics_df,
                     errorbar='se',
                     linewidth=LINEWIDTH,
                     ax = ax,
                     palette=color_map)

        # add horizontal line for untrained models
        sns.lineplot(x='percentage',
                     y=metric,
                     hue='downsampling_method',
                     data=untrained_baseline_df,
                     errorbar='se',
                     linewidth=LINEWIDTH,
                     ax = ax,
                     palette=color_map)
        
        # add horizontal lines for baselines
        if evaluation == "finetune" and task == "classification":
            # add line for logistic regression
            ax.hlines(y=logistic_regression_baseline, xmin=0, xmax=100, color='black', linestyle='dotted',
                      label='Logistic Regression\nBaseline', linewidth=LINEWIDTH)
        if evaluation == "finetune" and task == "perturbation":
            # add line for logistic regression
            ax.hlines(y=perturbation_baseline, xmin=0, xmax=100, color='black', linestyle='dotted',
                      label='No Change Baseline', linewidth=LINEWIDTH)
        if evaluation == "zeroshot":
            # add line for HVGs
            ax.hlines(y=hvg_baseline, xmin=0, xmax=100, color='black', linestyle='dotted', label='HVG Baseline', linewidth=LINEWIDTH)
            # add line for PCA
            ax.hlines(y=pca_baseline, xmin=0, xmax=100, color='black', linestyle='dashed', label='PCA Baseline', linewidth=LINEWIDTH)
        
        # set axes and legend
        ax.grid(False)
        ax.set_xlabel("")
        ax.set_ylabel("")

        ax.set_xlim([-1, 100])
        ax.set_ylim([y_lim_low, 1.0])

        if model == 'PretrainedPCA':
            ax.title.set_text('Pre-trained PCA')
        else:
            ax.title.set_text(model)

        ax.get_legend().remove() # have one legend for all models

        # remove top and right spines from axis
        ax.spines[['right', 'top']].set_visible(False)


        # set axis line widths
        for axis in ['bottom', 'left']:
            ax.spines[axis].set_linewidth(LINEWIDTH)


    if evaluation == "zeroshot":
        title = "Zero-Shot"
    if evaluation == "finetune":
        title = "Fine-Tuned"

    fig.suptitle(title + suffix, fontsize=XLARGE_FONT_SIZE, y=1.01)

    fig.supxlabel("Percentage of Full Pre-Training Dataset", y=0.2)

    if y_label == "$R^2$":
        offset = 0.08
    else:
        offset = 0.07
    fig.supylabel(y_label, y = 0.25 + 0.375, x=offset)

    # grab legend labels from subplot
    handles, labels = axes[0].get_legend_handles_labels()

    # untrained is duplicated by default, so use this trick of using a dict to remove duplicates
    by_label = dict(zip(labels, handles))

    # add legend blow plots
    fig.subplots_adjust(bottom=0.35)
    #fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, 0), title='Downsampling Method', ncol=5)
    fig.legend(by_label.values(), by_label.keys(), loc='lower center', bbox_to_anchor=(0.5, 0), title='', ncol=7)

    plot_file = f"figures/lineplots/{dataset}_{evaluation}_{task}{suffix}.svg"
    print("Saving:", plot_file)
    plt.savefig(plot_file, bbox_inches="tight")

# Figure 2: Plot Micro F1 score for hematopoiesis dataset

In [ ]:
models = ["PretrainedPCA", "scVI", "SSL", "Geneformer", "SCimilarity"]
dataset = "hematopoiesis"
include_spikeins = False
include_downsampling_schemes = True

plot_main_metric(models, dataset, "zeroshot", "classification", include_spikeins, include_downsampling_schemes)

plot_main_metric(models, dataset, "finetune", "classification", include_spikeins, include_downsampling_schemes)



# Figure 3: Plot AVG BIO score for Kim lung dataset

In [ ]:
models = ["PretrainedPCA", "scVI", "SSL", "Geneformer", "SCimilarity"]
dataset = "kim_lung"
include_spikeins = False
include_downsampling_schemes = True

plot_main_metric(models, dataset, "zeroshot", "integration", include_spikeins, include_downsampling_schemes, y_lim_low=0.2, suffix=" Batch Integration")


In [ ]:
def metrics_line_plots(evaluation, task, dataset, columns_to_plot, ylabels, title,
                       include_downsampling_methods=True, include_spikeins=False):
    print(evaluation, task, dataset)

    models = ["PretrainedPCA", "scVI", "SSL", "Geneformer", "SCimilarity"]
    formatted_models = ["Pre-trained\nPCA", "scVI", "SSL", "Geneformer", "SCimilarity"]

    if include_spikeins: # PCA and SCimilarity weren't trained with spikeins
        models = ["scVI", "SSL", "Geneformer"]
        formatted_models = ["scVI", "SSL", "Geneformer"]
    
    # load baseline(s)
    if evaluation == "finetune" and task == "classification":
        logistic_regression_baseline_df = pd.read_csv(f"metrics_csvs/finetune_classification_logistic_regression_variable_genes_baselines.csv", index_col=0)
        logistic_regression_baseline_df = logistic_regression_baseline_df[logistic_regression_baseline_df.dataset == dataset]

    if evaluation == "finetune" and task == "perturbation":
        noprediction_baseline_df = pd.read_csv(f"metrics_csvs/finetune_perturbation_noprediction_baseline.csv", index_col=0)
        noprediction_baseline_df = noprediction_baseline_df[noprediction_baseline_df.dataset == dataset]


    if evaluation == "zeroshot" and task == "classification":
        hvg_baseline_df = pd.read_csv(f"metrics_csvs/zero_shot_classification_variable_genes_baselines.csv", index_col=0)
        pca_baseline_df = pd.read_csv(f"metrics_csvs/zero_shot_classification_pca_baselines.csv", index_col=0)

    if evaluation == "zeroshot" and task == "integration":
        hvg_baseline_df = pd.read_csv(f"metrics_csvs/zero_shot_integration_variable_genes_baselines.csv", index_col=0)
        pca_baseline_df = pd.read_csv(f"metrics_csvs/zero_shot_integration_pca_baselines.csv", index_col=0)


    if evaluation == "zeroshot":
        hvg_baseline_df = hvg_baseline_df[hvg_baseline_df.dataset == dataset]
        pca_baseline_df = pca_baseline_df[pca_baseline_df.dataset == dataset]

    # plotting
    LINEWIDTH = 3

    sns.set_style("ticks")

    if not include_spikeins:
        fig, axes = plt.subplots(len(models), len(columns_to_plot), figsize=(30, 25), sharey=False)
    else: # PCA and SCimilarity weren't trained with spikeins
        models = ["scVI", "SSL", "Geneformer"]
        formatted_models = ["scVI", "SSL", "Geneformer"]
        # shrink plotting size because we are only plotting 3 models
        fig, axes = plt.subplots(len(models), len(columns_to_plot), figsize=(30, 0.6*25), sharey=False)

    for j, var in enumerate(columns_to_plot):
        for i, model in enumerate(models):
            metrics_df = pd.read_csv(f"metrics_csvs/{model}_{evaluation}_{task}_eval_results.csv", index_col=0)

            # subset to dataset of interest
            metrics_df = metrics_df[metrics_df["dataset"] == dataset]

            # exclude coresets
            metrics_df = metrics_df[metrics_df.downsampling_method != "coresets"]

            # if including spikeins subset to remove downsampling schemes for spikein supplemental figures
            if include_spikeins:
                metrics_df = metrics_df[metrics_df.downsampling_method != "geometric_sketching"]
                metrics_df = metrics_df[metrics_df.downsampling_method != "celltype_reweighted"]
    
            # subset to remove spikeins for main text figures
            if not include_spikeins:
                metrics_df = metrics_df[metrics_df.downsampling_method != "spikein_10"]
                metrics_df = metrics_df[metrics_df.downsampling_method != "spikein_50"]
        
    
    
            # rename categories for plotting
            metrics_df["downsampling_method"] = metrics_df["downsampling_method"].replace({
                'randomsplits': 'Random', 
                'random': 'Random', 
                'celltype_reweighted': 'Cell Type Reweighted',
                'geometric_sketch': "Geometric Sketching",
                'geometric_sketching': "Geometric Sketching",
                'spikein_10': "Spike-in (10%)",
                'spikein_50': "Spike-in (50%)",
                })

            # process 0pcts differently
            # put the untrained results in their own df
            untrained_baseline_df = metrics_df[metrics_df.percentage == 0]
            metrics_df = metrics_df[metrics_df.percentage != 0]
    
            # make it so that untrained results line will go across whole plot
            untrained_baseline_df_copy = untrained_baseline_df.copy()
            untrained_baseline_df_copy['percentage'] = 100
            untrained_baseline_df = pd.concat([untrained_baseline_df, untrained_baseline_df_copy], axis=0)
            untrained_baseline_df['downsampling_method'] = "Non pre-trained"
            
            # add the untrained results back to the original df
            metrics_df = pd.concat([metrics_df, untrained_baseline_df], axis=0)

            

            # default matplotlib colors:
            colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
            
            color_map = {
                'Random': colors[2],
                'Cell Type Reweighted': colors[0],
                'Geometric Sketching': colors[1],
                'Spike-in (10%)': colors[3],
                'Spike-in (50%)': colors[4],
                'Non pre-trained': colors[5]
            }
        
            sns.lineplot(x='percentage',
                         y=var,
                         hue='downsampling_method',
                         data=metrics_df,
                         errorbar='se',
                         marker='o',
                         ax = axes[i,j],
                         linewidth=LINEWIDTH,
                         palette=color_map)
            axes[i,j].grid(False)
            axes[i,j].set_xlabel("")
            axes[i,j].set_ylabel(ylabels[j])
            axes[i,j].legend(title='Downsampling Method')
            axes[i,j].set_xlim([0, 100])
            axes[i,j].set_ylim([0.0, 1.0])
    
            axes[i,j].get_legend().remove() # have one legend for all models
    
    
            # remove top and right spines from axis
            axes[i,j].spines[['right', 'top']].set_visible(False)
    
            # set axis line widths
            for axis in ['bottom', 'left']:
                axes[i,j].spines[axis].set_linewidth(LINEWIDTH)

    
    
            # add horizontal lines for baselines
            if evaluation == "finetune" and task == "classification":
                logistic_regression_baseline = logistic_regression_baseline_df[var][0]
                # add line for logistic regression
                axes[i,j].hlines(y=logistic_regression_baseline, xmin=0, xmax=100, color='black', linestyle='dotted',
                                 label='Logistic Regression\nBaseline', linewidth=LINEWIDTH)
            if evaluation == "finetune" and task == "perturbation":
                noprediction_baseline = noprediction_baseline_df[var][0]
                axes[i,j].hlines(y=noprediction_baseline, xmin=0, xmax=100, color='black', linestyle='dotted', label='No Change Baseline', linewidth=LINEWIDTH)
            if evaluation == "zeroshot":
                hvg_baseline = hvg_baseline_df[var][0]
                pca_baseline = pca_baseline_df[var][0]
    
                # add line for HVGs
                axes[i,j].hlines(y=hvg_baseline, xmin=0, xmax=100, color='black', linestyle='dotted', label='HVG Baseline', linewidth=LINEWIDTH)
                # add line for PCA
                axes[i,j].hlines(y=pca_baseline, xmin=0, xmax=100, color='black', linestyle='dashed', label='PCA Baseline', linewidth=LINEWIDTH)

            
    # grab legend labels from subplot
    handles, labels = axes[0,0].get_legend_handles_labels()

    # add legend blow plots
    fig.subplots_adjust(bottom=0.2)
    fig.legend(handles,
               labels,
               loc='lower center',
               bbox_to_anchor=(0.5, 0),
               title='Downsampling Method',
               ncol=7,
               fontsize=24,
               title_fontsize=28)

    fig.supxlabel("Percentage of Full Pre-Training Dataset", y=0.12, fontsize=32)

    fig.suptitle(title, fontsize=40)

    # todo add untrained baseline

    if not include_spikeins:
        # Add row titles
        vertical_intercept = 0.10
        vertical_spacing = 0.14
        fig.text(0.01, vertical_intercept + 5 * vertical_spacing, formatted_models[0], va='center', rotation='horizontal', fontsize=32)
        fig.text(0.01, vertical_intercept + 4 * vertical_spacing, formatted_models[1], va='center', rotation='horizontal', fontsize=32)
        fig.text(0.01, vertical_intercept + 3 * vertical_spacing, formatted_models[2], va='center', rotation='horizontal', fontsize=32)
        fig.text(0.01, vertical_intercept + 2 * vertical_spacing, formatted_models[3], va='center', rotation='horizontal', fontsize=32)
        fig.text(0.01, vertical_intercept + 1 * vertical_spacing, formatted_models[4], va='center', rotation='horizontal', fontsize=32)
    else:
        # Add row titles
        vertical_intercept = 0.07
        vertical_spacing = 0.23
        fig.text(0.01, vertical_intercept + 3 * vertical_spacing, formatted_models[0], va='center', rotation='horizontal', fontsize=32)
        fig.text(0.01, vertical_intercept + 2 * vertical_spacing, formatted_models[1], va='center', rotation='horizontal', fontsize=32)
        fig.text(0.01, vertical_intercept + 1 * vertical_spacing, formatted_models[2], va='center', rotation='horizontal', fontsize=32)

    plot_file = f"figures/lineplots_metrics/{dataset}_{evaluation}_{task}.svg"
    if include_spikeins:
        plot_file = f"figures/lineplots_metrics/{dataset}_{evaluation}_{task}_spikeins.svg"

    print("Saving:", plot_file)
    plt.savefig(plot_file, bbox_inches="tight") # todo bump this to 1200
    

def zero_shot_integration_plot(dataset, dataset_name, include_downsampling_methods=True, include_spikeins=False):
    columns_to_plot = ["NMI_cluster/label", "ARI_cluster/label", "ASW_label", "ASW_batch", "avg_bio"]
    ylabels = ["NMI (cluster/label)", "ARI (cluster/label)", "ASW (label)", "ASW (batch)", "AVG BIO"]
    title = f"Zero-Shot Integration: {dataset_name.capitalize()}"
    metrics_line_plots("zeroshot", "integration", dataset, columns_to_plot, ylabels, title,
                       include_downsampling_methods, include_spikeins)


def zero_shot_classification_plot(dataset, dataset_name, include_downsampling_methods=True, include_spikeins=False):
    columns_to_plot = ["accuracy", "precision", "recall", "micro_f1", "macro_f1"]
    ylabels = ["Accuracy", "Precision", "Recall", "Micro F1 Score", "Macro F1 Score"]
    title = f"Zero-Shot Classification: {dataset_name.capitalize()}"
    metrics_line_plots("zeroshot", "classification", dataset, columns_to_plot, ylabels, title,
                       include_downsampling_methods, include_spikeins)


def finetuned_classification_plot(dataset, dataset_name, include_downsampling_methods=True, include_spikeins=False):
    columns_to_plot = ["accuracy", "precision", "recall", "micro_f1", "macro_f1"]
    ylabels = ["Accuracy", "Precision", "Recall", "Micro F1 Score", "Macro F1 Score"]
    title = f"Fine-Tune Classification: {dataset_name.capitalize()}"
    metrics_line_plots("finetune", "classification", dataset, columns_to_plot, ylabels, title,
                       include_downsampling_methods, include_spikeins)


def finetuned_perturbation_plot(dataset, dataset_name, include_downsampling_methods=True, include_spikeins=False):
    columns_to_plot = ["R2", "mse"]
    ylabels = ["$R^2$", "MSE", "Recall"]
    title = f"Fine-Tune Perturbation: {dataset_name}"
    metrics_line_plots("finetune", "perturbation", dataset, columns_to_plot, ylabels, title,
                       include_downsampling_methods, include_spikeins)

# Supplemental Figures: Zero-Shot Classification (all metrics)

In [ ]:
datasets = ["hematopoiesis", "intestine", "periodontitis", "placenta"]

for dataset in datasets:
    dataset_name = dataset.capitalize()
    
    if dataset == "intestine":
        dataset_name = "Intestine-on-Chip"
        
    zero_shot_classification_plot(dataset, dataset_name)

# Supplemental Figures: Fine-Tuned Classification (all metrics)

In [ ]:
for dataset in ["hematopoiesis", "intestine", "periodontitis", "placenta"]:
    dataset_name = dataset.capitalize()
    
    if dataset == "intestine":
        dataset_name = "Intestine-on-Chip"
        
    finetuned_classification_plot(dataset, dataset_name)

# Supplemental Figures: Zero-Shot Integration (all metrics)

In [ ]:
for dataset in ["kim_lung", "liver", "periodontitis", "renal"]:
    if dataset == "kim_lung":
        dataset_name = "Lung"
    else:
        dataset_name = dataset.capitalize()
    zero_shot_integration_plot(dataset, dataset_name)

# Supplemental Figures: Spike-In Zero-Shot Classification (all metrics)

In [ ]:
datasets = ["hematopoiesis", "intestine", "periodontitis", "placenta"]

for dataset in datasets:
    dataset_name = dataset.capitalize()
    
    if dataset == "intestine":
        dataset_name = "Intestine-on-Chip"
        
    zero_shot_classification_plot(dataset, dataset_name, include_downsampling_methods=False, include_spikeins=True)


# Supplemental Figures: Spike-In Fine-Tune Classification (all metrics)

In [ ]:
datasets = ["hematopoiesis", "intestine", "periodontitis", "placenta"]

for dataset in datasets:
    dataset_name = dataset.capitalize()
    
    if dataset == "intestine":
        dataset_name = "Intestine-on-Chip"
        
    finetuned_classification_plot(dataset, dataset_name, include_downsampling_methods=False, include_spikeins=True)


# Supplemental Figures: Spike-In Zero-Shot Integration (all metrics)

In [ ]:
for dataset in ["kim_lung", "periodontitis", "renal"]:
    if dataset == "kim_lung":
        dataset_name = "Lung"
    else:
        dataset_name = dataset.capitalize()
    zero_shot_integration_plot(dataset, dataset_name, include_downsampling_methods=False, include_spikeins=True)

In [ ]:
def tmp(evaluation, task, dataset, columns_to_plot, ylabels, title,
                       include_downsampling_methods=True, include_spikeins=False):
    print(evaluation, task, dataset)

    models = ["PretrainedPCA", "scVI", "SSL", "SCimilarity"]
    formatted_models = ["Pretrained\nPCA", "scVI", "SSL", "Geneformer", "SCimilarity"]

    if include_spikeins: # PCA and SCimilarity weren't trained with spikeins
        models = ["scVI", "SSL", "Geneformer"]
        formatted_models = ["scVI", "SSL", "Geneformer"]
    
    # load baseline(s)
    if evaluation == "finetune" and task == "classification":
        logistic_regression_baseline_df = pd.read_csv(f"metrics_csvs/finetune_classification_logistic_regression_variable_genes_baselines.csv", index_col=0)
        logistic_regression_baseline_df = logistic_regression_baseline_df[logistic_regression_baseline_df.dataset == dataset]

        logistic_regression_baseline = logistic_regression_baseline_df["micro_f1"][0]

    if evaluation == "zeroshot" and task == "classification":
        hvg_baseline_df = pd.read_csv(f"metrics_csvs/zero_shot_classification_variable_genes_baselines.csv", index_col=0)
        pca_baseline_df = pd.read_csv(f"metrics_csvs/zero_shot_classification_pca_baselines.csv", index_col=0)

    if evaluation == "zeroshot" and task == "integration":
        hvg_baseline_df = pd.read_csv(f"metrics_csvs/zero_shot_integration_variable_genes_baselines.csv", index_col=0)
        pca_baseline_df = pd.read_csv(f"metrics_csvs/zero_shot_integration_pca_baselines.csv", index_col=0)


    if evaluation == "zeroshot":
        hvg_baseline_df = hvg_baseline_df[hvg_baseline_df.dataset == dataset]
        pca_baseline_df = pca_baseline_df[pca_baseline_df.dataset == dataset]

    # plotting
    LINEWIDTH = 3

    sns.set_style("ticks")

    if not include_spikeins:
        fig, axes = plt.subplots(len(models), len(columns_to_plot), figsize=(30, 25), sharey=False)
    else: # PCA and SCimilarity weren't trained with spikeins
        models = ["scVI", "SSL", "Geneformer"]
        formatted_models = ["scVI", "SSL", "Geneformer"]
        # shrink plotting size because we are only plotting 3 models
        fig, axes = plt.subplots(len(models), len(columns_to_plot), figsize=(30, 0.6*25), sharey=False)

    for j, var in enumerate(columns_to_plot):
        for i, model in enumerate(models):
            metrics_df = pd.read_csv(f"metrics_csvs/{model}_{evaluation}_{task}_eval_results.csv", index_col=0)

            # subset to dataset of interest
            metrics_df = metrics_df[metrics_df["dataset"] == dataset]

            # if including spikeins subset to remove downsampling schemes for spikein supplemental figures
            if include_spikeins:
                metrics_df = metrics_df[metrics_df.downsampling_method != "geometric_sketching"]
                metrics_df = metrics_df[metrics_df.downsampling_method != "celltype_reweighted"]
    
            # subset to remove spikeins for main text figures
            if not include_spikeins:
                metrics_df = metrics_df[metrics_df.downsampling_method != "spikein_10"]
                metrics_df = metrics_df[metrics_df.downsampling_method != "spikein_50"]
        
    
    
            # rename categories for plotting
            metrics_df["downsampling_method"] = pd.Categorical(metrics_df["downsampling_method"]).rename_categories({
                'randomsplits': 'Random', 
                'random': 'Random', 
                'celltype_reweighted': 'Cell Type Reweighted',
                'geometric_sketch': "Geometric Sketching",
                'geometric_sketching': "Geometric Sketching",
                'spikein_10': "Spike-in (10%)",
                'spikein_50': "Spike-in (50%)",
                })

            # process 0pcts differently
            # put the untrained results in their own df
            untrained_baseline_df = metrics_df[metrics_df.percentage == 0]
            metrics_df = metrics_df[metrics_df.percentage != 0]
    
            # make it so that untrained results line will go across whole plot
            untrained_baseline_df_copy = untrained_baseline_df.copy()
            untrained_baseline_df_copy['percentage'] = 100
            untrained_baseline_df = pd.concat([untrained_baseline_df, untrained_baseline_df_copy], axis=0)
            untrained_baseline_df['downsampling_method'] = "Non pre-trained"
            
            # add the untrained results back to the original df
            metrics_df = pd.concat([metrics_df, untrained_baseline_df], axis=0)

            

            # default matplotlib colors:
            colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
            
            color_map = {
                'Random': colors[2],
                'Cell Type Reweighted': colors[0],
                'Geometric Sketching': colors[1],
                'Spike-in (10%)': colors[3],
                'Spike-in (50%)': colors[4],
                'Non pre-trained': colors[5]
            }
        
            sns.lineplot(x='percentage',
                         y=var,
                         hue='downsampling_method',
                         data=metrics_df,
                         errorbar='se',
                         marker='o',
                         ax = axes[i,j],
                         linewidth=LINEWIDTH,
                         palette=color_map)
            axes[i,j].grid(False)
            axes[i,j].set_xlabel("")
            axes[i,j].set_ylabel(ylabels[j])
            axes[i,j].legend(title='Downsampling Method')
            axes[i,j].set_xlim([0, 100])
            axes[i,j].set_ylim([0.0, 1.0])
    
            axes[i,j].get_legend().remove() # have one legend for all models
    
    
            # remove top and right spines from axis
            axes[i,j].spines[['right', 'top']].set_visible(False)
    
            # set axis line widths
            for axis in ['bottom', 'left']:
                axes[i,j].spines[axis].set_linewidth(LINEWIDTH)

    
    
            # add horizontal lines for baselines
            if evaluation == "finetune":
                # add line for logistic regression
                axes[i,j].hlines(y=logistic_regression_baseline, xmin=0, xmax=100, color='black', linestyle='dotted',
                                 label='Logistic Regression\nBaseline', linewidth=LINEWIDTH)
            if evaluation == "zeroshot":
                hvg_baseline = hvg_baseline_df[var][0]
                pca_baseline = pca_baseline_df[var][0]
    
                # add line for HVGs
                axes[i,j].hlines(y=hvg_baseline, xmin=0, xmax=100, color='black', linestyle='dotted', label='HVG Baseline', linewidth=LINEWIDTH)
                # add line for PCA
                axes[i,j].hlines(y=pca_baseline, xmin=0, xmax=100, color='black', linestyle='dashed', label='PCA Baseline', linewidth=LINEWIDTH)

            
    # grab legend labels from subplot
    handles, labels = axes[0,0].get_legend_handles_labels()

    # add legend blow plots
    fig.subplots_adjust(bottom=0.2)
    fig.legend(handles,
               labels,
               loc='lower center',
               bbox_to_anchor=(0.5, 0),
               title='Downsampling Method',
               ncol=7,
               fontsize=24,
               title_fontsize=28)

    fig.supxlabel("Percentage of Full Pre-Training Dataset", y=0.12, fontsize=32)

    fig.suptitle(title, fontsize=40)

    # todo add untrained baseline

    if not include_spikeins:
        # Add row titles
        vertical_intercept = 0.10
        vertical_spacing = 0.14
        fig.text(0.01, vertical_intercept + 5 * vertical_spacing, formatted_models[0], va='center', rotation='horizontal', fontsize=32)
        fig.text(0.01, vertical_intercept + 4 * vertical_spacing, formatted_models[1], va='center', rotation='horizontal', fontsize=32)
        fig.text(0.01, vertical_intercept + 3 * vertical_spacing, formatted_models[2], va='center', rotation='horizontal', fontsize=32)
        fig.text(0.01, vertical_intercept + 2 * vertical_spacing, formatted_models[3], va='center', rotation='horizontal', fontsize=32)
        fig.text(0.01, vertical_intercept + 1 * vertical_spacing, formatted_models[4], va='center', rotation='horizontal', fontsize=32)
    else:
        # Add row titles
        vertical_intercept = 0.07
        vertical_spacing = 0.23
        fig.text(0.01, vertical_intercept + 3 * vertical_spacing, formatted_models[0], va='center', rotation='horizontal', fontsize=32)
        fig.text(0.01, vertical_intercept + 2 * vertical_spacing, formatted_models[1], va='center', rotation='horizontal', fontsize=32)
        fig.text(0.01, vertical_intercept + 1 * vertical_spacing, formatted_models[2], va='center', rotation='horizontal', fontsize=32)

    plot_file = f"figures/lineplots_metrics/{dataset}_{evaluation}_{task}.svg"
    if include_spikeins:
        plot_file = f"figures/lineplots_metrics/{dataset}_{evaluation}_{task}_spikeins.svg"

    print("Saving:", plot_file)
    plt.savefig(plot_file, bbox_inches="tight") # todo bump this to 1200
    

def zero_shot_integration_plot2(dataset, dataset_name, include_downsampling_methods=True, include_spikeins=False):
    columns_to_plot = ["NMI_cluster/label", "ARI_cluster/label", "ASW_label", "ASW_batch", "avg_bio"]
    ylabels = ["NMI (cluster/label)", "ARI (cluster/label)", "ASW (label)", "ASW (batch)", "AVG BIO"]
    title = f"Zero-Shot Integration: {dataset_name.capitalize()}"
    metrics_line_plots2("zeroshot", "integration", dataset, columns_to_plot, ylabels, title,
                       include_downsampling_methods, include_spikeins)


In [ ]:
"""
for dataset in ["ocular"]:
    if dataset == "kim_lung":
        dataset_name = "Lung"
    else:
        dataset_name = dataset.capitalize()
    zero_shot_integration_plot2(dataset, dataset_name)
"""

# Perturbation fine tuning

In [ ]:
models = ["PretrainedPCA", "scVI", "SSL", "Geneformer", "SCimilarity"]
dataset = "tak901" # dinaciclib, homoharringtonine, ph797804, tak901
include_spikeins = False
include_downsampling_schemes = True

for dataset in ['dinaciclib', 'homoharringtonine', 'ph797804', 'tak901']:

    plot_main_metric(models, dataset, "finetune", "perturbation", include_spikeins, include_downsampling_schemes, y_lim_low=0.5, suffix=" Perturbation Prediction (held out cell type)")



In [ ]:
datasets = ['dinaciclib', 'homoharringtonine', 'ph797804', 'tak901']
dataset_names = ['Dinaciclib', 'Homoharringtonine', 'PH-797804', 'TAK-901']

for i in range(len(datasets)):
    dataset = datasets[i]
    dataset_name = dataset_names[i]
    finetuned_perturbation_plot(dataset, dataset_name, include_downsampling_methods=True, include_spikeins=False)